In [ ]:

import pandas as pd
import itertools
import re
from openpyxl import load_workbook
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import files

print("PENGUMPULAN FILE EXCEL DIALEKTOLOGI")
print("Anda bisa mengunggah file secara bertahap dari folder yang berbeda.")

uploaded_files = []

while True:
    print("\nSilakan pilih file Excel (.xlsx) untuk diunggah:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename not in uploaded_files:
            uploaded_files.append(filename)
            print(f" -> Berhasil menambahkan: {filename}")
        else:
            print(f" -> File {filename} sudah ada di dalam daftar.")

    lanjut = input("\nApakah Anda ingin menambahkan file lagi dari folder lain? (y/n): ").strip().lower()
    if lanjut != 'y':
        break

if len(uploaded_files) < 2:
    print("\n[Peringatan] Anda membutuhkan minimal 2 file untuk melakukan pembandingan. Silakan jalankan ulang cell ini.")
else:
    print(f"\nMemproses {len(uploaded_files)} file...")

    START_ROW = 8
    END_ROW = 1089
    COL_KODE = 2
    COL_BERIAN = 4

    # 2. Fungsi Bantuan Dialektometri
    def levenshtein_distance(a, b):
        if a == b: return 0
        if len(a) < len(b): a, b = b, a
        previous = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            current = [i]
            for j, cb in enumerate(b, 1):
                insert = current[j - 1] + 1
                delete = previous[j] + 1
                replace = previous[j - 1] + (ca != cb)
                current.append(min(insert, delete, replace))
            previous = current
        return previous[-1]

    def get_lexemes(word_string):
        if pd.isna(word_string) or word_string is None:
            return []
        w = str(word_string).strip().lower()
        w = w.replace("[", "").replace("]", "")
        lexemes = [lex.strip() for lex in re.split(r'[;,/]', w) if lex.strip()]
        return lexemes

    def format_kode(raw_kode):
        if isinstance(raw_kode, float) and raw_kode.is_integer():
            return str(int(raw_kode))
        return str(raw_kode).strip()

    def read_dialect_data(filepath):
        try:
            wb = load_workbook(filepath, data_only=True)
            ws = wb.active
            data = {}
            for row in range(START_ROW, min(END_ROW, ws.max_row) + 1):
                raw_kode = ws.cell(row=row, column=COL_KODE).value
                berian = ws.cell(row=row, column=COL_BERIAN).value
                if raw_kode:
                    kode_str = format_kode(raw_kode)
                    data[kode_str] = berian
            return data
        except Exception as e:
            print(f"Gagal membaca {filepath}: {e}")
            return {}

    def klasifikasi_guiter(persentase):
        p = round(persentase)
        if p <= 20: return "Tidak ada perbedaan"
        elif p <= 30: return "Beda wicara"
        elif p <= 50: return "Beda subdialek"
        elif p <= 80: return "Beda dialek"
        else: return "Beda bahasa"

    def klasifikasi_lauder(persentase):
        p = round(persentase)
        if p <= 30: return "Tidak ada perbedaan"
        elif p <= 40: return "Beda wicara"
        elif p <= 50: return "Beda subdialek"
        elif p <= 69: return "Beda dialek"
        else: return "Beda bahasa"

    # LOGIKA PENGKATEGORIAN DINAMIS
    def get_kategori_konsep(kode_str):
        k = str(kode_str).strip().upper()

        # 1. Pengecekan Khusus Swadesh (1001-1200)
        try:
            num = int(k)
            if 1001 <= num <= 1200:
                return "Swadesh (1001-1200)"
        except ValueError:
            pass

        # 2. Pengecekan 2 Karakter Pertama untuk Kategori Lainnya
        if len(k) >= 2:
            return f"Kategori {k[:2]}"
        elif len(k) == 1:
            return f"Kategori {k}"
        else:
            return "Kategori Lainnya"

    # 3. Ekstraksi Data
    print("Mengekstrak data dari file Excel...")
    all_data = {}
    tp_mapping = {}

    for i, file in enumerate(uploaded_files):
        alias_tp = f"TP {i+1}"
        nama_asli = file.replace(".xlsx", "").replace("FIX_2026_", "")
        tp_mapping[alias_tp] = nama_asli
        all_data[alias_tp] = read_dialect_data(file)

    semua_kode = sorted(list(set(itertools.chain.from_iterable(d.keys() for d in all_data.values()))))
    daerah_list = list(all_data.keys())
    pasangan_daerah = list(itertools.combinations(daerah_list, 2))

    kategori_map = {}
    for kode in semua_kode:
        kat = get_kategori_konsep(kode)
        if kat not in kategori_map:
            kategori_map[kat] = []
        kategori_map[kat].append(kode)

    # 4. Antarmuka Pemilihan Konsep
    print("\n=== PEMILIHAN KATEGORI KONSEP ===")
    pilihan_kategori = ["Semua Konsep (Gabungan)"] + sorted(list(kategori_map.keys()))

    dropdown_kategori = widgets.SelectMultiple(
        options=pilihan_kategori,
        value=["Semua Konsep (Gabungan)"],
        description='Kategori:',
        disabled=False,
        layout=widgets.Layout(width='60%', height='180px') # Tinggi ditambah agar muat lebih banyak kategori
    )

    btn_proses = widgets.Button(
        description='Hitung Dialektometri',
        button_style='primary',
        icon='calculator',
        layout=widgets.Layout(width='200px', margin='10px 0px 0px 0px')
    )

    output_area = widgets.Output()

    def jalankan_perhitungan(b):
        with output_area:
            clear_output(wait=True)

            kategori_terpilih = dropdown_kategori.value
            kode_target = set()

            if "Semua Konsep (Gabungan)" in kategori_terpilih:
                kode_target = set(semua_kode)
                label_pilihan = "Semua Konsep"
            else:
                for kat in kategori_terpilih:
                    kode_target.update(kategori_map[kat])
                label_pilihan = ", ".join(kategori_terpilih)

            kode_target = sorted(list(kode_target))
            print(f"\nMemproses {len(kode_target)} konsep untuk: {label_pilihan}...\n")

            hasil_perbandingan = []
            rekap_pasangan = {f"{d1} ↔ {d2}": {"total_banding": 0, "total_beda": 0} for d1, d2 in pasangan_daerah}

            for kode in kode_target:
                for d1, d2 in pasangan_daerah:
                    w1_raw = all_data[d1].get(kode, "")
                    w2_raw = all_data[d2].get(kode, "")
                    pasangan_key = f"{d1} ↔ {d2}"

                    lex1 = get_lexemes(w1_raw)
                    lex2 = get_lexemes(w2_raw)

                    if not lex1 or not lex2:
                        continue

                    min_dist = float('inf')
                    min_ratio = 100.0
                    best_pair = ("", "")

                    for l1 in lex1:
                        for l2 in lex2:
                            dist = levenshtein_distance(l1, l2)
                            max_len = max(len(l1), len(l2))
                            ratio = (dist / max_len) * 100 if max_len > 0 else 0

                            if ratio < min_ratio:
                                min_ratio = ratio
                                min_dist = dist
                                best_pair = (l1, l2)

                    if min_ratio == 0:
                        status = "Identik"
                    elif min_ratio <= 50:
                        status = "Variasi Fonetis"
                    else:
                        status = "Beda Leksikal"
                        rekap_pasangan[pasangan_key]["total_beda"] += 1

                    rekap_pasangan[pasangan_key]["total_banding"] += 1

                    row_data = {
                        "Kategori Konsep": get_kategori_konsep(kode),
                        "Kode Konsep": kode,
                        "Pasangan": pasangan_key,
                    }
                    row_data[f"Data {d1}"] = f"[{w1_raw.strip()}]" if w1_raw else "Ø"
                    row_data[f"Data {d2}"] = f"[{w2_raw.strip()}]" if w2_raw else "Ø"

                    info_polimorfi = f"[{best_pair[0]}] vs [{best_pair[1]}]" if (len(lex1)>1 or len(lex2)>1) else "-"
                    row_data["Pasangan Dievaluasi"] = info_polimorfi
                    row_data["Jarak"] = min_dist
                    row_data["Beda Kata (%)"] = round(min_ratio, 2)
                    row_data["Status"] = status

                    hasil_perbandingan.append(row_data)

            if not hasil_perbandingan:
                print("Tidak ada data yang dapat dibandingkan untuk kategori yang dipilih.")
                return

            hasil_ringkasan = []
            for pasangan, rekap in rekap_pasangan.items():
                tot_band = rekap["total_banding"]
                tot_beda = rekap["total_beda"]
                pct_beda = (tot_beda / tot_band) * 100 if tot_band > 0 else 0

                hasil_ringkasan.append({
                    "Pasangan TP": pasangan,
                    "Total Kosakata Dibandingkan": tot_band,
                    "Total Kosakata Berbeda": tot_beda,
                    "Persentase Perbedaan Leksikal (%)": round(pct_beda, 2),
                    "Skala Guiter (1973)": klasifikasi_guiter(pct_beda),
                    "Skala Lauder": klasifikasi_lauder(pct_beda)
                })

            df_mapping = pd.DataFrame(list(tp_mapping.items()), columns=["Kode TP", "Nama Asli Daerah"])
            df_detail = pd.DataFrame(hasil_perbandingan).fillna("Ø (Nihil)")

            statis = ["Pasangan Dievaluasi", "Jarak", "Beda Kata (%)", "Status"]
            kolom_dinamis = [c for c in df_detail.columns if c not in statis]
            df_detail = df_detail[kolom_dinamis + statis]

            df_ringkasan = pd.DataFrame(hasil_ringkasan)

            nama_file_output = "Hasil_Menjajal_Dialektometri.xlsx"
            with pd.ExcelWriter(nama_file_output, engine='openpyxl') as writer:
                df_mapping.to_excel(writer, sheet_name='Keterangan TP', index=False)
                df_ringkasan.to_excel(writer, sheet_name='Ringkasan Klasifikasi', index=False)
                df_detail.to_excel(writer, sheet_name='Detail Perbandingan', index=False)

            display(HTML("<b>KETERANGAN TITIK PENGAMATAN (TP)</b>"))
            display(df_mapping)

            display(HTML("<br><b>RINGKASAN KLASIFIKASI KESELURUHAN</b>"))
            display(df_ringkasan)

            display(HTML("<br><b>PREVIEW DETAIL PEMBANDINGAN (Preview dibatasi 50 baris agar browser tidak lag)</b>"))
            with pd.option_context('display.max_rows', 50):
                display(df_detail)

            def on_download_clicked(b):
                files.download(nama_file_output)

            download_btn = widgets.Button(description='Unduh File Excel', button_style='success', icon='download', layout=widgets.Layout(width='200px'))
            download_btn.on_click(on_download_clicked)

            display(HTML("<br><b>Unduh hasil perhitungan khusus segmentasi ini (Seluruh baris komplit ada di dalam Excel):</b>"))
            display(download_btn)

    btn_proses.on_click(jalankan_perhitungan)

    display(HTML("<span style='font-size:12px; color:gray;'>Gunakan <b>Ctrl+Click</b> (Windows) atau <b>Cmd+Click</b> (Mac) untuk memilih lebih dari satu kategori.</span>"))
    display(dropdown_kategori, btn_proses, output_area)

PENGUMPULAN FILE EXCEL DIALEKTOLOGI
Anda bisa mengunggah file secara bertahap dari folder yang berbeda.

Silakan pilih file Excel (.xlsx) untuk diunggah:


Saving FIX_2026_GALANG BARU.xlsx to FIX_2026_GALANG BARU.xlsx
Saving FIX_2026_REMPANG CATE.xlsx to FIX_2026_REMPANG CATE.xlsx
Saving Pangkil_Data Pemetaan Lama.xlsx to Pangkil_Data Pemetaan Lama.xlsx
Saving Pulau Abang_Data Pemetaan Lama.xlsx to Pulau Abang_Data Pemetaan Lama.xlsx
Saving Pulau Karas_Data Pemetaan Lama.xlsx to Pulau Karas_Data Pemetaan Lama.xlsx
 -> Berhasil menambahkan: FIX_2026_GALANG BARU.xlsx
 -> Berhasil menambahkan: FIX_2026_REMPANG CATE.xlsx
 -> Berhasil menambahkan: Pangkil_Data Pemetaan Lama.xlsx
 -> Berhasil menambahkan: Pulau Abang_Data Pemetaan Lama.xlsx
 -> Berhasil menambahkan: Pulau Karas_Data Pemetaan Lama.xlsx

Apakah Anda ingin menambahkan file lagi dari folder lain? (y/n): n

Memproses 5 file...
Mengekstrak data dari file Excel...

=== PEMILIHAN KATEGORI KONSEP ===


SelectMultiple(description='Kategori:', index=(0,), layout=Layout(height='180px', width='60%'), options=('Semu…

Button(button_style='primary', description='Hitung Dialektometri', icon='calculator', layout=Layout(margin='10…

Output()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>